# 🔗 LangChain Chains with a Local Model — Explained

**What this notebook does:** builds a LangChain chain around an *open-source* model — Mistral-7B, loaded locally in 4-bit — instead of a hosted API like Cohere. It logs into Hugging Face, loads and quantizes the model, wraps it for LangChain, then builds a prompt template and chain that explains a given scientific process. Each cell below has a short explanation directly above it.

### 📦 Cell 0 — Install the libraries

`langchain-huggingface` lets LangChain talk to Hugging Face models, and `bitsandbytes` is what makes 4-bit quantized loading possible in Cell 3 — needed here because this notebook runs an actual open-source LLM locally, instead of calling a hosted API like Cohere the way the previous notebook did.

In [ ]:
!pip install langchain cohere langchain-huggingface bitsandbytes langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.9 MB/s eta 0:00:00


### 🔑 Cell 1 — Log in to Hugging Face

Reads a Hugging Face access token out of Colab's secret storage and uses it to authenticate. This is necessary because Mistral-7B (loaded in Cell 3) is a **gated model** — Hugging Face requires you to be logged in, and to have accepted its usage terms, before it will let you download it.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

### 🧰 Cell 2 — Import the tools

`transformers` and its classes (`AutoTokenizer`, `AutoModelForCausalLM`) are Hugging Face's library for loading models directly, rather than calling them through an API. `BitsAndBytesConfig` configures 4-bit quantization — shrinking the model so it fits in less GPU memory. `HuggingFacePipeline` is the LangChain wrapper that lets a locally-loaded Hugging Face model slot into a chain the same way `ChatCohere` did in the previous notebook.

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer,BitsAndBytesConfig,AutoModelForCausalLM
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_huggingface.llms import HuggingFacePipeline

### 🦙 Cell 3 — Load Mistral-7B in 4-bit, and wrap it for LangChain

This is the biggest cell, so here's what each part does:

- `model_name` picks **Mistral-7B**, a 7-billion-parameter open-source model.
- `tokenizer` turns text into the tokens the model understands; padding is set to the right side, reusing the end-of-sequence token as the pad token.
- `bnb_config` sets up **4-bit quantization** — the same core idea as the vector compression from the vector database notes, but applied here to shrink the *model's own weights*, not embeddings, so a 7B-parameter model can fit in far less GPU memory.
- `AutoModelForCausalLM.from_pretrained(...)` actually downloads and loads the model with that quantization applied.
- `text_generation_pipeline` wraps the model and tokenizer into a ready-to-use text-generation pipeline, with `temperature=0.1` (low randomness, more predictable output) and a cap of 512 new tokens per response.
- `HuggingFacePipeline(...)` wraps that whole pipeline so LangChain can use it as `mistral_llm` — exactly the way `ChatCohere` was used in the previous notebook's `LLMChain`.

**Example:** this is the same overall pattern as calling a hosted API model, just self-hosted instead — `mistral_llm` can be dropped into an `LLMChain` exactly like `ChatCohere` was, only now the model runs locally on this machine's GPU instead of over the network.

In [ ]:
model_name='mistralai/Mistral-7B-v0.1'

model_config = transformers.AutoConfig.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

use_4bit = True

bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)

text_generation_pipeline = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=512,
    output_scores=True
)

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['output_scores']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'output_scores', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Sequential Chain

### 🔗 Cell 5 — Build a new prompt template and chain

Same pattern as the previous notebook: `template` has a placeholder, `{topic}`, this time asking the model to act as an expert explaining a given scientific process. `PromptTemplate` wraps it, and `LLMChain` combines it with `mistral_llm` (the locally-loaded model from Cell 3) into a reusable chain called `chain`.

In [ ]:
template = """
I want you to act as a expert who can generate details on the given scientific process {topic}.
"""

prompt_template = PromptTemplate(
    input_variables=["topic"],
    template=template,
)

chain = LLMChain(llm=mistral_llm, prompt=prompt_template)

/tmp/ipykernel_865/3555638293.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=mistral_llm, prompt=prompt_template)


### 👀 Cell 6 — Preview the filled-in prompt, without running the model

`prompt_template.format(...)` just substitutes `{topic}` with `"Photosynthesis"` and returns the resulting text — useful for checking exactly what prompt would be sent, before actually calling the model in the next cell.

In [ ]:
description = "Photosynthesis"
prompt_template.format(topic=description)

'\nI want you to act as a expert who can generate details on the given scientific process Photosynthesis.\n'

### ▶️ Cell 7 — Run the chain for real

`.invoke(...)` takes the same `topic` value, fills it into the prompt automatically, sends it to Mistral-7B, and prints the model's generated explanation of photosynthesis.

In [ ]:
print(chain.invoke(input={'topic':description}))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'topic': 'Photosynthesis', 'text': '\nI want you to act as a expert who can generate details on the given scientific process Photosynthesis.\n\nThe process of photosynthesis is a process that is carried out by plants and other organisms. It is a process that is used to convert light energy into chemical energy. The process of photosynthesis is a process that is used to convert light energy into chemical energy.\n\nThe process of photosynthesis is a process that is used to convert light energy into chemical energy. The process of photosynthesis is a process that is used to convert light energy into chemical energy.\n\nThe process of photosynthesis is a process that is used to convert light energy into chemical energy. The process of photosynthesis is a process that is used to convert light energy into chemical energy.\n\nThe process of photosynthesis is a process that is used to convert light energy into chemical energy. The process of photosynthesis is a process that is used to conver